In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout, Masking
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
FEATURE_DIR = '/kaggle/input/datasets/ganuwoahh/cholec80-i3d-features-real'

def load_sequences(video_ids):
    X, y = [], []
    for vid in video_ids:
        feature_path = os.path.join(FEATURE_DIR, f'video{vid:02d}_features.npy')
        label_path = os.path.join(FEATURE_DIR, f'video{vid:02d}_labels.npy')
        
        if os.path.exists(feature_path) and os.path.exists(label_path):
            X.append(np.load(feature_path))
            y.append(np.argmax(np.load(label_path), axis=1)) # converting one hot back into integers
            
    X_padded = pad_sequences(X, padding='post', dtype='float32', value=0.0) # PADDING
    y_padded = pad_sequences(y, padding='post', dtype='int32', value=-1) # this will kill the metrics but we'll make our own metrics later
    
    return X_padded, y_padded

X_train, y_train = load_sequences(range(1, 71))
X_val, y_val = load_sequences(range(71, 81))
print(f"Training Data Shape: {X_train.shape}")

In [ ]:
model = Sequential([
    tf.keras.Input(shape=(None, 1024)), # 1024 dimension feature vector
    
    Masking(mask_value=0.0), # ignore padding
    
    Bidirectional(LSTM(256, return_sequences=True, dropout=0.3)),
    Bidirectional(LSTM(128, return_sequences=True, dropout=0.3)),
    
    # FINAL CLASSIFIER
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(7, activation='softmax')
])

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(ignore_class=-1)

model.compile(optimizer='adam', 
              loss=loss_fn, 
              metrics=['accuracy']) # again fake accuracy don't worry about this

model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30, # can train for more epochs since it's so light i didn't find much difference between 20 30 and 50
    batch_size=4
)

In [ ]:
model.save('/kaggle/working/bilstm_cholec80.keras')

In [ ]:
# f1 score and cm

y_pred_probs = model.predict(X_val)
y_pred_classes = np.argmax(y_pred_probs, axis=-1)

y_true_flat = y_val.flatten()
y_pred_flat = y_pred_classes.flatten()

real_data_mask = y_true_flat != -1 # this drops the -1 padding we used to make everything the same length

y_true_real = y_true_flat[real_data_mask]
y_pred_real = y_pred_flat[real_data_mask]

phase_names = [
    "0: Preparation", "1: Calot Triangle Dissection", "2: Clipping & Cutting",
    "3: Gallbladder Dissection", "4: Gallbladder Packaging", "5: Cleaning & Coagulation", "6: Gallbladder Retrieval"
]

report = classification_report(y_true_real, y_pred_real, target_names=phase_names, zero_division=0)
print(report)

cm = confusion_matrix(y_true_real, y_pred_real)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
cm_normalized = np.nan_to_num(cm_normalized)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="Blues", 
            xticklabels=phase_names, yticklabels=phase_names)
plt.title("BiLSTM Normalized Confusion Matrix")
plt.ylabel("True Surgical Phase")
plt.xlabel("Predicted Surgical Phase")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# roc auc

y_pred_probs_flat = y_pred_probs.reshape(-1, 7)
y_pred_probs_real = y_pred_probs_flat[real_data_mask]

plt.figure(figsize=(12, 8))
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown', 'pink']

for i in range(7):
    true_binary = (y_true_real == i).astype(int) 
    if np.sum(true_binary) > 0:
        fpr, tpr, _ = roc_curve(true_binary, y_pred_probs_real[:, i])
        roc_auc = auc(fpr, tpr)
        
        plt.plot(fpr, tpr, color=colors[i], lw=2,
                 label=f'{phase_names[i]} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Guessing (AUC = 0.500)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('BiLSTM Multi-Class ROC Curves')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/roc_curves_bilstm.png', dpi=300)
plt.show()